# 03 — Feature Engineering

Transform raw gameweek data into model-ready features. The golden rule: **no data leakage**.
Every feature must use only information available *before* the gameweek we're predicting.

Features we'll build:
1. Rolling averages (3-GW, 5-GW) for points, xG, xA, minutes, bonus
2. Fixture difficulty rating (FDR)
3. Home/away indicator
4. Player price and price momentum
5. Season-to-date aggregates
6. Rest days between matches
7. Target variable: next gameweek's points

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

plt.style.use('ggplot')
pd.set_option('display.max_columns', 40)

## 1. Load and prepare base data

In [ ]:
BOOTSTRAP_PATH = project_root / 'data' / 'raw' / 'bootstrap_static.json'
FIXTURES_PATH = project_root / 'data' / 'raw' / 'fixtures.json'
HISTORIES_DIR = project_root / 'data' / 'raw' / 'player_histories'

with open(BOOTSTRAP_PATH) as f:
    bootstrap = json.load(f)
with open(FIXTURES_PATH) as f:
    fixtures_raw = json.load(f)

players_meta = pd.DataFrame(bootstrap['elements'])
teams = pd.DataFrame(bootstrap['teams'])
fixtures = pd.DataFrame(fixtures_raw)

pos_map = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
team_map = dict(zip(teams['id'], teams['name']))

players_meta['position'] = players_meta['element_type'].map(pos_map)
players_meta['team_name'] = players_meta['team'].map(team_map)

# Load all gameweek histories
rows = []
for hist_file in HISTORIES_DIR.glob('*.json'):
    pid = int(hist_file.stem.split('_')[1])
    with open(hist_file) as f:
        data = json.load(f)
    for gw in data['history']:
        gw['player_id'] = pid
        rows.append(gw)

gw_df = pd.DataFrame(rows)

# Join metadata
gw_df = gw_df.merge(
    players_meta[['id', 'web_name', 'position', 'team_name', 'element_type', 'team']],
    left_on='player_id', right_on='id', how='left'
)

# Type conversions
float_cols = ['expected_goals', 'expected_assists', 'expected_goal_involvements',
              'expected_goals_conceded', 'influence', 'creativity', 'threat', 'ict_index']
for col in float_cols:
    gw_df[col] = gw_df[col].astype(float)

gw_df['price'] = gw_df['value'] / 10
gw_df['kickoff_time'] = pd.to_datetime(gw_df['kickoff_time'])
gw_df = gw_df.sort_values(['player_id', 'round']).reset_index(drop=True)

print(f'Base dataset: {len(gw_df):,} rows, {gw_df["player_id"].nunique()} players, GW{gw_df["round"].min()}-{gw_df["round"].max()}')

## 2. Rolling features (lagged)

For each player, compute rolling averages over the **previous** N gameweeks.
We use `.shift(1)` before `.rolling()` to ensure no leakage from the current GW.

In [ ]:
rolling_cols = {
    'total_points': 'pts',
    'minutes': 'min',
    'goals_scored': 'goals',
    'assists': 'ast',
    'bonus': 'bonus',
    'bps': 'bps',
    'expected_goals': 'xg',
    'expected_assists': 'xa',
    'expected_goal_involvements': 'xgi',
    'expected_goals_conceded': 'xgc',
    'clean_sheets': 'cs',
    'influence': 'infl',
    'creativity': 'crea',
    'threat': 'thrt',
    'ict_index': 'ict',
}

windows = [3, 5]

for col, short in rolling_cols.items():
    # Shift first to avoid leakage
    shifted = gw_df.groupby('player_id')[col].shift(1)
    for w in windows:
        gw_df[f'{short}_roll{w}'] = (
            shifted.groupby(gw_df['player_id'])
            .transform(lambda x: x.rolling(w, min_periods=1).mean())
        )

print(f'Added {len(rolling_cols) * len(windows)} rolling features')
print(f'Example columns: {[c for c in gw_df.columns if "roll" in c][:6]}')

## 3. Season-to-date averages (lagged)

Expanding mean of key stats up to (but not including) the current GW.

In [ ]:
season_cols = ['total_points', 'minutes', 'expected_goals', 'expected_assists', 'bonus']

for col in season_cols:
    short = rolling_cols.get(col, col)
    shifted = gw_df.groupby('player_id')[col].shift(1)
    gw_df[f'{short}_season_avg'] = (
        shifted.groupby(gw_df['player_id'])
        .transform(lambda x: x.expanding(min_periods=1).mean())
    )

# Games played so far (before this GW)
gw_df['games_played'] = gw_df.groupby('player_id').cumcount()

print(f'Added {len(season_cols)} season-to-date features + games_played')

## 4. Fixture difficulty rating (FDR)

In [ ]:
# FDR from fixtures data — difficulty of the opponent
# Build a lookup: for each team in each GW, what's the difficulty?
fdr_rows = []
for _, fix in fixtures.iterrows():
    if pd.isna(fix.get('event')):
        continue
    gw = int(fix['event'])
    # Home team faces away difficulty, away team faces home difficulty
    fdr_rows.append({'team': fix['team_h'], 'round': gw, 'fdr': fix['team_h_difficulty'], 'is_home': True})
    fdr_rows.append({'team': fix['team_a'], 'round': gw, 'fdr': fix['team_a_difficulty'], 'is_home': False})

fdr_df = pd.DataFrame(fdr_rows)

# Merge FDR into main dataframe
gw_df = gw_df.merge(
    fdr_df[['team', 'round', 'fdr']],
    left_on=['team', 'round'], right_on=['team', 'round'],
    how='left'
)

print(f'FDR distribution:')
print(gw_df['fdr'].value_counts().sort_index())

## 5. Price momentum

Price change over the last few gameweeks — reflects market sentiment.

In [ ]:
gw_df['price_prev'] = gw_df.groupby('player_id')['price'].shift(1)
gw_df['price_change_1gw'] = gw_df['price'] - gw_df['price_prev']

price_3gw = gw_df.groupby('player_id')['price'].shift(3)
gw_df['price_change_3gw'] = gw_df['price'] - price_3gw

print('Price change (1 GW) distribution:')
print(gw_df['price_change_1gw'].describe().round(3))

## 6. Rest days between matches

In [ ]:
gw_df['prev_kickoff'] = gw_df.groupby('player_id')['kickoff_time'].shift(1)
gw_df['rest_days'] = (gw_df['kickoff_time'] - gw_df['prev_kickoff']).dt.days

print('Rest days distribution:')
print(gw_df['rest_days'].describe().round(1))

## 7. Define target variable

The target is the **current** gameweek's `total_points`. All features above are lagged,
so this is safe — we're predicting this GW's points using only prior information.

In [ ]:
gw_df['target'] = gw_df['total_points']

print(f'Target (total_points) distribution:')
print(gw_df['target'].describe().round(2))

## 8. Assemble final feature matrix

Select only the features we'll use for modeling, drop rows with NaN from early GWs.

In [ ]:
# Feature columns
rolling_features = [c for c in gw_df.columns if 'roll' in c or 'season_avg' in c]
other_features = ['was_home', 'fdr', 'price', 'price_change_1gw', 'price_change_3gw',
                  'rest_days', 'games_played', 'element_type']
feature_cols = rolling_features + other_features

id_cols = ['player_id', 'web_name', 'position', 'team_name', 'round']

model_df = gw_df[id_cols + feature_cols + ['target']].copy()

# Drop rows where we don't have enough history (first 2 GWs per player)
model_df = model_df[model_df['round'] >= 3].dropna(subset=feature_cols)

print(f'Final feature matrix: {model_df.shape[0]:,} rows x {len(feature_cols)} features')
print(f'Gameweeks: GW{model_df["round"].min()} to GW{model_df["round"].max()}')
print(f'Players: {model_df["player_id"].nunique()}')
print(f'\nFeature list ({len(feature_cols)}):')
for f in feature_cols:
    print(f'  {f}')

## 9. Feature correlation with target

Which engineered features best predict GW points?

In [ ]:
corr = model_df[feature_cols + ['target']].corr()['target'].drop('target').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
corr.plot(kind='barh', ax=ax)
ax.set_xlabel('Correlation with GW Points')
ax.set_title('Feature Correlations with Target')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('\nTop 15 features by |correlation|:')
print(corr.abs().sort_values(ascending=False).head(15).round(3).to_string())

## 10. Save feature matrix for modeling

In [ ]:
output_dir = project_root / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'features.csv'
model_df.to_csv(output_path, index=False)

print(f'Saved feature matrix to {output_path}')
print(f'Shape: {model_df.shape}')
print(f'File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB')